# 02a · Train all classifiers (v2: mixed data, self-blended fakes, shared degradation)

Trains every model in `MODELS` with **the same v2 recipe**:
- **Data:** FF++ (01a), Celeb-DF train/val (01b) and the real photos FFHQ + CelebA-HQ (01c), mixed in fixed
  proportions per epoch: **40% real**, **35% self-blended fakes** (made on the fly from real crops with cached
  landmarks), **25% the datasets' own fakes**. Each dataset gets the same share within its group, and an epoch is
  `SAMPLES_PER_EPOCH` images however much data is attached.
- **Augmentation:** flip, colour jitter, light random erasing, and **the same quality degradation for real and fake
  images** (blur, downscale-upscale, JPEG, noise, sharpening), so image quality no longer predicts the label.
- 224×224 input, AdamW + OneCycle, up to 8 epochs, early stopping on the pooled validation AUC.
- After training, each best checkpoint scores the validation sets and saves `val_preds.npz` + `train_feats.npy`
  for calibration (02c). StyleGAN / diffusion faces are **not** trained on here (they train the CLIP probe, 02b),
  but they are scored on validation so 02c can fit the CNN + CLIP fusion.

| Model | Why it's in the comparison |
|---|---|
| `efficientnet_b0` | proposed model: small, fast, Grad-CAM++-friendly |
| `meso4` | MesoNet-4: small purpose-built deepfake CNN (Afchar et al. 2018), FF++ paper baseline, trained from scratch |
| `legacy_xception` | Xception: the FaceForensics++ paper's reference detector |
| `resnet50` | standard CNN baseline |
| `efficientnet_b4` | larger EfficientNet (DFDC-winning family) |
| `vit_small_patch16_224` | Vision Transformer (ViT-S/16) |
| `convnext_tiny` | ConvNeXt-T: modern CNN redesigned with transformer-era ideas (Liu et al. 2022) |

**Time:** an epoch is `SAMPLES_PER_EPOCH = 60000` images, about twice a v1 epoch, so all seven models take
roughly **12–14 GPU-hours** on a T4 (B0 ~80 min, Meso ~55 min, ResNet / ViT ~1.7 h, ConvNeXt ~2 h, Xception ~2.4 h,
B4 ~3.3 h; early stopping can shorten them). Kaggle stops a session at 12 h and **loses the output if it times
out**, so this notebook skips models that won't fit in `TIME_BUDGET_H`. Make a **copy** and split `MODELS` between
the two: copy A = B0, Meso, ResNet, ViT (~6 h); copy B = Xception, B4, ConvNeXt (~7.5 h). 02c and 03 pick up the
models from all attached copies.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On
- **Inputs:** outputs of **01a** (FF++), **01b** (Celeb-DF with train/val) and **01c** (HQ photos + landmarks)

**Produces:** `/kaggle/working/outputs/checkpoints/` (EfficientNet-B0 in `best.pt`, every other model in
`<model>/best.pt`, each with `history.json`, `val_preds.npz` and `train_feats.npy`).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, find_datasets, find_landmarks
setup(data=False)

import json, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

ALL = find_datasets()
LANDMARKS = find_landmarks()
TRAIN_SETS = {k: v for k, v in ALL.items() if k != "DFDC"}   # DFDC is test-only
print("datasets:", TRAIN_SETS)
print("landmarks:", LANDMARKS, sorted(p.name for p in LANDMARKS.glob("*.npz")) if LANDMARKS else "")
assert "FF++" in TRAIN_SETS, "Add the output of 01a (FF++) as an input"
assert LANDMARKS, "Add the output of 01c (landmarks/) as an input: no self-blended fakes without it"
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## Settings

In [ ]:
MODELS = ["efficientnet_b0", "meso4", "resnet50", "vit_small_patch16_224", "legacy_xception", "efficientnet_b4",
          "convnext_tiny"]
EPOCHS, BATCH, PATIENCE, WEIGHT_DECAY, SUBSET = 8, 32, 3, 0.05, None
SAMPLES_PER_EPOCH = 60000
VAL_MAX_PER_SET = 5000    # keep equal to 02b's, so the CNN and CLIP validation lists line up for fusion
SYNTHETIC_SHARE = 0.0     # > 0 also trains the CNNs on StyleGAN / diffusion fakes
LR = {"default": 3e-4, "meso4": 1e-3, "vit_small_patch16_224": 1e-4, "convnext_tiny": 1e-4}   # from-scratch Meso needs more, ViT less
TIME_BUDGET_H = 11.0

# Rough training cost relative to EfficientNet-B0, used to skip models that would overrun the session
RELATIVE_COST = {"efficientnet_b0": 1.0, "meso4": 0.7, "resnet50": 1.3, "vit_small_patch16_224": 1.3,
                 "legacy_xception": 1.8, "efficientnet_b4": 2.5, "convnext_tiny": 1.5}
DATA_ARGS = " ".join(f'--data "{k}={v}"' for k, v in TRAIN_SETS.items()) + f' --landmarks "{LANDMARKS}"'

## What a training batch looks like
The training mix as the model sees it: real crops, self-blended fakes and dataset fakes, after augmentation.
Self-blended fakes should be hard to tell from their real originals.

In [ ]:
from src.classifier.dataset import MixedTrainSet, load_landmarks
from src.config import MEAN, STD

shares = {"real": 0.4, "sbi": 0.35, "fake": 0.25 - SYNTHETIC_SHARE, "synthetic": SYNTHETIC_SHARE}
preview = MixedTrainSet(TRAIN_SETS, load_landmarks(LANDMARKS, TRAIN_SETS), shares=shares)
print(preview.describe())
rng = np.random.default_rng(0)
idx = rng.choice(len(preview), 16, p=(preview.weights / preview.weights.sum()).numpy())
fig, axes = plt.subplots(2, 8, figsize=(17, 4.6))
for ax, i in zip(axes.ravel(), idx):
    x, y = preview[i]
    img = (x.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{preview.items[i][2]} · {preview.items[i][3]}\nlabel {'fake' if y else 'real'}", fontsize=8)
plt.tight_layout(); plt.show()
del preview

## Train

In [ ]:
start, unit_h, log = time.time(), None, []
for m in MODELS:
    elapsed_h = (time.time() - start) / 3600
    estimate_h = unit_h * RELATIVE_COST.get(m, 2.0) if unit_h else 0
    if elapsed_h + estimate_h > TIME_BUDGET_H:
        print(f"SKIP {m}: ~{estimate_h:.1f} h needed, {TIME_BUDGET_H - elapsed_h:.1f} h left. Train it in a copy of this notebook.")
        log.append({"model": m, "status": "skipped"}); continue
    print(f"\n{'=' * 30} {m} {'=' * 30}")
    t0 = time.time()
    args = (f"--backbone {m} --epochs {EPOCHS} --batch-size {BATCH} --lr {LR.get(m, LR['default'])} "
            f"--patience {PATIENCE} --weight-decay {WEIGHT_DECAY} --workers 4 --samples-per-epoch {SAMPLES_PER_EPOCH} "
            f"--val-max-per-set {VAL_MAX_PER_SET} --synthetic-share {SYNTHETIC_SHARE} {DATA_ARGS}")
    if SUBSET: args += f" --subset {SUBSET}"
    !TQDM_MININTERVAL=60 python -m src.classifier.train {args}
    hours = (time.time() - t0) / 3600
    unit_h = unit_h or hours / RELATIVE_COST.get(m, 2.0)
    log.append({"model": m, "status": "done", "hours": round(hours, 2)})
display(pd.DataFrame(log))

## Training curves (all models)
`val_auc` pools the validation sets of the datasets the CNNs train on. `auc_<dataset>` is per dataset;
`fpr_<dataset>` is the share of real photos called fake (real-only sets) at the validation threshold.

In [ ]:
ck = Path("/kaggle/working/outputs/checkpoints")
histories = {("efficientnet_b0" if h.parent == ck else h.parent.name): pd.read_json(h) for h in ck.rglob("history.json")}
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
for m, h in histories.items():
    a.plot(h.epoch, h.train_loss, marker="o", label=m)
    b.plot(h.epoch, h.val_auc, marker="o", label=m)
a.set_title("train loss"); b.set_title("pooled validation AUC")
for ax in (a, b): ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

best = pd.DataFrame([{"model": m, "best_epoch": int(h.loc[h.val_auc.idxmax(), "epoch"]),
                      **h.loc[h.val_auc.idxmax()].drop(["epoch", "train_loss", "elapsed_min"]).to_dict(),
                      "minutes": h.elapsed_min.iloc[-1]} for m, h in histories.items()]).sort_values("val_auc", ascending=False)
display(best.round(4))
!ls -la /kaggle/working/outputs/checkpoints /kaggle/working/outputs/checkpoints/*/ 2>/dev/null | grep -E "best.pt|val_preds|train_feats|history"